In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np

In [4]:
sold = pd.read_csv('/Users/morganstevenson/Desktop/IDX/sold_cleaned.csv')
listings = pd.read_csv('/Users/morganstevenson/Desktop/IDX/listing_cleaned.csv')

/var/folders/ch/xc77p32n37v6pfqh60_ls62c0000gn/T/ipykernel_55860/3110500308.py:1: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv('/Users/morganstevenson/Desktop/IDX/sold_cleaned.csv')
/var/folders/ch/xc77p32n37v6pfqh60_ls62c0000gn/T/ipykernel_55860/3110500308.py:2: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  listings = pd.read_csv('/Users/morganstevenson/Desktop/IDX/listing_cleaned.csv')


In [12]:
# Columns to be converted to date time
date_cols = ['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']

# Apply pd.to_datetime across all specified columns for both listing and sold data frames
listings[date_cols] = listings[date_cols].apply(pd.to_datetime)
sold[date_cols] = sold[date_cols].apply(pd.to_datetime)

In [15]:
sold['PriceRatio'] = sold['ClosePrice'] / sold['OriginalListPrice']
sold['PricePerSqFt'] = sold['ClosePrice'] / sold['LivingArea']
sold['Year'] = sold['CloseDate'].dt.year
sold['Month'] = sold['CloseDate'].dt.month
sold['YrM'] = sold['CloseDate'].dt.to_period('M').astype(str)
sold['ListingtoContractDays'] = sold['PurchaseContractDate'] - sold['ListingContractDate']
sold['ContracttoCloseDays'] = sold['CloseDate'] - sold['PurchaseContractDate']

In [16]:
listings['PriceRatio'] = listings['ClosePrice'] / listings['OriginalListPrice']
listings['PricePerSqFt'] = listings['ClosePrice'] / listings['LivingArea']
listings['Year'] = listings['CloseDate'].dt.year
listings['Month'] = listings['CloseDate'].dt.month
listings['YrM'] = listings['CloseDate'].dt.to_period('M').astype(str)
listings['ListingtoContractDays'] = listings['PurchaseContractDate'] - listings['ListingContractDate']
listings['ContracttoCloseDays'] = listings['CloseDate'] - listings['PurchaseContractDate']

In [17]:
districts = gpd.read_file('/Users/morganstevenson/Desktop/IDX/week 6/DistrictAreas2526_-284845464123469011.geojson')
districts.head()

,OBJECTID,Year,FedID,CDCode,CDSCode,CountyName,DistrictName,DistrictType,GradeLow,GradeHigh,...,MIGcount,MIGpct,SWDcount,SWDpct,SEDcount,SEDpct,DistrctAreaSqMi,LocaleCode,LocaleDesc,geometry
0,1,2025-26,0601770,0161119,01611190000000,Alameda,Alameda Unified,Unified,PK,12,...,0,0.0,1302,12.1,4259,39.5,11.248886,21,"21 - Suburban, Large","MULTIPOLYGON (((-13606222.82 4540862.699, -136..."
1,2,2025-26,0601860,0161127,01611270000000,Alameda,Albany City Unified,Unified,PK,12,...,0,0.0,363,9.7,1247,33.3,1.789975,21,"21 - Suburban, Large","POLYGON ((-13612893.866 4565099.707, -13612896..."
2,3,2025-26,0604740,0161143,01611430000000,Alameda,Berkeley Unified,Unified,PK,12,...,0,0.0,1118,11.9,2710,28.8,10.434281,12,"12 - City, Midsize","POLYGON ((-13609482.48 4565074.597, -13609483...."
3,4,2025-26,0607800,0161150,01611500000000,Alameda,Castro Valley Unified,Unified,PK,12,...,2,0.0,1186,12.2,3784,39.0,66.885261,21,"21 - Suburban, Large","MULTIPOLYGON (((-13582508.535 4529067.071, -13..."
4,5,2025-26,0612630,0161168,01611680000000,Alameda,Emery Unified,Unified,PK,12,...,0,0.0,98,16.1,407,66.7,1.273923,21,"21 - Suburban, Large","POLYGON ((-13613999.038 4555592.769, -13614126..."


In [18]:
unified = districts[districts["DistrictType"] == "Unified"].copy()

print(len(unified))

345


In [21]:
unified = unified.to_crs("EPSG:4326")

In [22]:
listing_gdf = gpd.GeoDataFrame(
    listings,
    geometry=gpd.points_from_xy(
        listings["Longitude"],
        listings["Latitude"]
    ),
    crs="EPSG:4326"
)

sold_gdf = gpd.GeoDataFrame(
    sold,
    geometry=gpd.points_from_xy(
        sold["Longitude"],
        sold["Latitude"]
    ),
    crs="EPSG:4326"
)

In [23]:
listing_with_district = gpd.sjoin(
    listing_gdf,
    unified[["DistrictName", "geometry"]],
    how="left",
    predicate="within"
)

sold_with_district = gpd.sjoin(
    sold_gdf,
    unified[["DistrictName", "geometry"]],
    how="left",
    predicate="within"
)

In [27]:
listings["DistrictName"] = listing_with_district["DistrictName"].values
sold["DistrictName"] = sold_with_district["DistrictName"].values

In [28]:
property_summary = (
    sold.groupby(["PropertyType", "PropertySubType"])
    .agg(
        Transactions=("ListingKey", "count"),
        MedianSalePrice=("ClosePrice", "median"),
        MeanSalePrice=("ClosePrice", "mean"),
        MedianDOM=("DaysOnMarket", "median"),
        MeanDOM=("DaysOnMarket", "mean"),
        MedianPPSF=("PricePerSqFt", "median"),
        MeanPPSF=("PricePerSqFt", "mean")
    )
    .reset_index()
)

property_summary.head()

,PropertyType,PropertySubType,Transactions,MedianSalePrice,MeanSalePrice,MedianDOM,MeanDOM,MedianPPSF,MeanPPSF
0,Residential,BoatSlip,84,182000.0,215896.428571,31.5,65.285714,1750.000000,2051.695652
1,Residential,Cabin,488,242750.0,291709.629098,49.5,80.502049,287.285748,604.479119
2,Residential,CoOwnership,18,400000.0,749694.388889,24.0,39.777778,411.651469,485.708763
3,Residential,Condominium,69238,625000.0,875664.199549,25.0,42.699154,563.157895,707.832528
4,Residential,DeededParking,6,577500.0,575833.333333,12.0,21.333333,348.407995,410.512905


In [29]:
market_summary = (
    sold.groupby(["CountyOrParish", "MLSAreaMajor"])
    .agg(
        Transactions=("ListingKey", "count"),
        MedianSalePrice=("ClosePrice", "median"),
        MeanSalePrice=("ClosePrice", "mean"),
        MedianDOM=("DaysOnMarket", "median"),
        MeanDOM=("DaysOnMarket", "mean"),
        MedianPPSF=("PricePerSqFt", "median"),
        MeanPPSF=("PricePerSqFt", "mean")
    )
    .reset_index()
)

market_summary.head()

,CountyOrParish,MLSAreaMajor,Transactions,MedianSalePrice,MeanSalePrice,MedianDOM,MeanDOM,MedianPPSF,MeanPPSF
0,Alameda,699 - Not Defined,1746,1153500.0,1.261970e+06,13.0,26.946735,725.130632,773.157702
1,Alameda,BERK - Berkeley,3,782999.0,5.743330e+05,12.0,27.666667,609.097918,576.712240
2,Amador,699 - Not Defined,4,332500.0,3.510000e+05,91.5,105.500000,153.033794,188.671134
3,Butte,699 - Not Defined,57,399000.0,4.175070e+05,37.0,65.421053,274.012964,252.455264
4,Butte,PARA - Paradise,59,384900.0,4.011218e+05,55.0,79.271186,271.935484,259.667858


In [30]:
listing_office_summary = (
    sold.groupby("ListOfficeName")
    .agg(
        Transactions=("ListingKey", "count"),
        MedianSalePrice=("ClosePrice", "median"),
        MeanSalePrice=("ClosePrice", "mean"),
        MedianDOM=("DaysOnMarket", "median"),
        MeanDOM=("DaysOnMarket", "mean"),
        MedianPPSF=("PricePerSqFt", "median")
    )
    .sort_values("Transactions", ascending=False)
    .reset_index()
)

listing_office_summary.head(20)

,ListOfficeName,Transactions,MedianSalePrice,MeanSalePrice,MedianDOM,MeanDOM,MedianPPSF
0,Compass,29013,1345000.0,1.820599e+06,15.0,31.917933,752.314815
1,Coldwell Banker Realty,18781,1181250.0,1.647109e+06,17.0,35.493903,690.399137
2,Keller Williams Realty,8323,875000.0,1.029332e+06,15.0,32.083263,539.898132
3,First Team Real Estate,6023,960000.0,1.125946e+06,13.0,30.050307,608.812746
4,Berkshire Hathaway HomeServices California Pro...,5477,950000.0,1.497217e+06,23.0,41.647070,599.220919
5,eXp Realty of California Inc,4953,785000.0,9.749023e+05,19.0,37.530789,529.446758
6,Real Broker,4930,840000.0,1.039686e+06,15.0,30.096349,556.225256
7,Intero Real Estate Services,4356,1380000.0,1.653167e+06,12.0,25.323232,848.656030
8,"eXp Realty of California, Inc.",3640,753500.0,9.234646e+05,13.0,28.330495,489.307127
9,Equity Union,3609,850000.0,1.059359e+06,29.0,45.672763,477.630114


In [31]:
buyer_office_summary = (
    sold.groupby("BuyerOfficeName")
    .agg(
        Transactions=("ListingKey", "count"),
        MedianSalePrice=("ClosePrice", "median"),
        MeanSalePrice=("ClosePrice", "mean"),
        MedianDOM=("DaysOnMarket", "median"),
        MeanDOM=("DaysOnMarket", "mean"),
        MedianPPSF=("PricePerSqFt", "median")
    )
    .sort_values("Transactions", ascending=False)
    .reset_index()
)

buyer_office_summary.head(20)

,BuyerOfficeName,Transactions,MedianSalePrice,MeanSalePrice,MedianDOM,MeanDOM,MedianPPSF
0,Compass,27095,1320000.0,1.783414e+06,16.0,33.360805,745.702080
1,Coldwell Banker Realty,14998,1185700.0,1.691747e+06,17.0,35.770836,686.558765
2,NONMEMBER MRML,9406,501957.5,6.343285e+05,26.0,48.622156,290.303707
3,Keller Williams Realty,6507,840000.0,1.258439e+06,18.0,35.458737,533.972681
4,Real Broker,6422,800000.0,1.015534e+06,18.0,33.748209,541.795666
5,First Team Real Estate,5406,900000.0,1.086284e+06,14.0,31.670551,597.391970
6,eXp Realty of California Inc,5406,830000.0,1.039329e+06,19.0,37.529412,568.747885
7,"eXp Realty of California, Inc.",4771,725000.0,9.781516e+05,18.0,34.503249,492.837143
8,Berkshire Hathaway HomeServices California Pro...,4254,980000.0,1.488694e+06,21.0,40.039257,609.409190
9,Intero Real Estate Services,3520,1330000.0,1.570082e+06,12.0,25.868750,809.098389
